# DermoAI — MobileNetV2 FST V-VI Skin Lesion Classifier v2

## Project Objective

This notebook implements an AI-assisted dermatological triage system optimized for Fitzpatrick Skin Types (FST) V-VI (darker skin tones) in resource-limited settings in Rwanda. The system performs multi-class classification of skin lesions and routes cases to either **REFER** (urgent specialist referral) or **MANAGE LOCALLY** (primary care management).

## 7 Skin Condition Classes

| Class | Triage Decision |
|-------|----------------|
| `lichen_planus` | **MANAGE LOCALLY** |
| `lupus_erythematosus` | **REFER** |
| `pityriasis_rubra_pilaris` | **REFER** |
| `psoriasis` | **MANAGE LOCALLY** |
| `scabies` | **MANAGE LOCALLY** |
| `squamous_cell_carcinoma` | **REFER** |
| `vitiligo` | **MANAGE LOCALLY** |

**REFER classes:** Conditions requiring specialist referral (lupus_erythematosus, pityriasis_rubra_pilaris, squamous_cell_carcinoma)

**MANAGE LOCALLY classes:** Conditions treatable at primary care level (lichen_planus, psoriasis, scabies, vitiligo)

## Two-Phase Training Strategy

**Phase 1 — Feature Extraction (20 epochs)**
- Base MobileNetV2 frozen (ImageNet weights preserved)
- Train classification head only
- Learning rate: 1e-3
- REFER class weight: 2.0x

**Phase 2 — Fine-Tuning (15 epochs)**
- Unfreeze last 30 layers of MobileNetV2
- Fine-tune with lower learning rate: 1e-5
- REFER class weight: 3.0x (increased penalty for missed referrals)

## Performance Targets

- **Overall Accuracy:** ≥80% (minimum: 70%)
- **REFER Recall:** ≥75% (critical for patient safety)
- **Minimum Class Recall:** ≥50% for all classes
- **Critical Errors:** ≤3 REFER cases misclassified as MANAGE LOCALLY
- **FST Equity Gap:** <10% between FST V and FST VI performance

## Two-Stage Triage Decision Logic

**Stage 1 — Classification:**
- MobileNetV2 predicts one of 7 named classes with confidence scores

**Stage 2 — Triage Routing:**
1. **Low Confidence Check:** If max confidence < 0.50 → **UNCERTAIN** → route to best REFER class
2. **REFER Safety Net:** If any REFER class probability > 0.35 → force REFER prediction
3. **Otherwise:** Apply class-based triage decision (REFER or MANAGE LOCALLY)

This conservative approach ensures patient safety by defaulting uncertain cases to specialist referral.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from tensorflow.keras.metrics import Precision, Recall, AUC

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score, f1_score
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

In [ ]:
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"Random seed set to: {RANDOM_SEED}")

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU detected. Training will use CPU.")

In [ ]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    DRIVE_BASE = Path('/content/drive/MyDrive/dermoai')
    LOCAL_BASE = Path('/content/dermoai_local')
    
    print("Copying data to local storage for faster training...")
    import shutil
    
    LOCAL_BASE.mkdir(parents=True, exist_ok=True)
    
    for split in ['train', 'val', 'test']:
        src_dir = DRIVE_BASE / 'data' / ('augmented' if split == 'train' else 'processed/fitzpatrick17k') / split
        dst_dir = LOCAL_BASE / 'data' / ('augmented' if split == 'train' else 'processed/fitzpatrick17k') / split
        
        if not dst_dir.exists() and src_dir.exists():
            print(f"  Copying {split} data...")
            shutil.copytree(src_dir, dst_dir)
    
    BASE_DIR = LOCAL_BASE
    print(f"Using local storage: {BASE_DIR}")
else:
    BASE_DIR = Path('../')
    print(f"Using workspace storage: {BASE_DIR}")

TRAIN_DIR = BASE_DIR / 'data' / 'augmented' / 'train'
VAL_DIR = BASE_DIR / 'data' / 'processed' / 'fitzpatrick17k' / 'val'
TEST_DIR = BASE_DIR / 'data' / 'processed' / 'fitzpatrick17k' / 'test'
MODEL_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results' / 'training'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("\nPaths configured:")
print(f"  TRAIN_DIR:   {TRAIN_DIR}")
print(f"  VAL_DIR:     {VAL_DIR}")
print(f"  TEST_DIR:    {TEST_DIR}")
print(f"  MODEL_DIR:   {MODEL_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")

In [ ]:
CONFIG = {
    'model': {
        'architecture': 'MobileNetV2',
        'input_shape': (224, 224, 3),
        'num_classes': 7,
        'weights': 'imagenet',
        'dropout_rate': 0.4
    },
    'training': {
        'batch_size': 32,
        'phase1_epochs': 20,
        'phase2_epochs': 15,
        'initial_lr': 1e-3,
        'finetune_lr': 1e-5,
        'use_class_weights': True
    },
    'callbacks': {
        'early_stopping_patience': 7,
        'reduce_lr_patience': 3,
        'reduce_lr_factor': 0.5,
        'min_lr': 1e-7
    },
    'evaluation': {
        'refer_classes': ['lupus_erythematosus', 'pityriasis_rubra_pilaris', 'squamous_cell_carcinoma'],
        'manage_locally_classes': ['lichen_planus', 'psoriasis', 'scabies', 'vitiligo'],
        'confidence_threshold': 0.50,
        'refer_recall_target': 0.75,
        'min_recall_threshold': 0.50,
        'fst_equity_gap_threshold': 0.10
    }
}

print("Configuration loaded:")
print(json.dumps(CONFIG, indent=2))

In [ ]:
CLASS_NAMES = sorted([
    'lichen_planus',
    'lupus_erythematosus',
    'pityriasis_rubra_pilaris',
    'psoriasis',
    'scabies',
    'squamous_cell_carcinoma',
    'vitiligo'
])

TRIAGE_MAPPING = {
    'lichen_planus':            'MANAGE LOCALLY',
    'lupus_erythematosus':      'REFER',
    'pityriasis_rubra_pilaris': 'REFER',
    'psoriasis':                'MANAGE LOCALLY',
    'scabies':                  'MANAGE LOCALLY',
    'squamous_cell_carcinoma':  'REFER',
    'vitiligo':                 'MANAGE LOCALLY',
    'UNCERTAIN':                'REFER',
}

REFER_CLASSES = CONFIG['evaluation']['refer_classes']

print("7 Skin Condition Classes and Triage Decisions:")
print("=" * 60)
for i, cls in enumerate(CLASS_NAMES):
    triage = TRIAGE_MAPPING[cls]
    print(f"{i}: {cls:30s} → {triage}")
print("=" * 60)
print(f"\nREFER classes: {REFER_CLASSES}")

In [ ]:
print("Data Distribution Verification")
print("=" * 80)

data_dist = []
for cls in CLASS_NAMES:
    train_count = len(list((TRAIN_DIR / cls).glob('*.jpg'))) if (TRAIN_DIR / cls).exists() else 0
    val_count = len(list((VAL_DIR / cls).glob('*.jpg'))) if (VAL_DIR / cls).exists() else 0
    test_count = len(list((TEST_DIR / cls).glob('*.jpg'))) if (TEST_DIR / cls).exists() else 0
    triage = TRIAGE_MAPPING[cls]
    
    data_dist.append({
        'Class': cls,
        'Train': train_count,
        'Val': val_count,
        'Test': test_count,
        'Triage': triage
    })

df_dist = pd.DataFrame(data_dist)
print(df_dist.to_string(index=False))
print("=" * 80)

for split in ['Train', 'Val', 'Test']:
    split_total = df_dist[split].sum()
    print(f"{split} total: {split_total}")

print("\nVerifying all 7 classes present in all splits...")
for split, split_dir in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    split_classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    assert split_classes == CLASS_NAMES, f"{split} split missing classes: expected {CLASS_NAMES}, got {split_classes}"
    print(f"  ✓ {split.upper()}: All 7 classes present")

train_counts = df_dist['Train'].values
imbalance_ratio = train_counts.max() / train_counts.min() if train_counts.min() > 0 else float('inf')
print(f"\nTrain imbalance ratio: {imbalance_ratio:.2f} (1.0 = perfectly balanced)")

In [ ]:
print("Building tf.data pipeline...")

BATCH_SIZE = CONFIG['training']['batch_size']
IMG_SIZE = (224, 224)

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=CLASS_NAMES,
    shuffle=True,
    seed=RANDOM_SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=CLASS_NAMES,
    shuffle=False,
    seed=RANDOM_SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=CLASS_NAMES,
    shuffle=False,
    seed=RANDOM_SEED
)

normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

print("\nClass Index Mapping:")
print("=" * 60)
for idx, cls in enumerate(CLASS_NAMES):
    triage = TRIAGE_MAPPING[cls]
    print(f"{idx}: {cls:30s} → {triage}")
print("=" * 60)

class_names_path = MODEL_DIR / 'class_names.json'
with open(class_names_path, 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)
print(f"\nClass names saved to: {class_names_path}")

In [ ]:
print("Computing Class Weights")
print("=" * 60)

class_weight_dict = {
    idx: 2.0 if cls in REFER_CLASSES else 1.0
    for idx, cls in enumerate(CLASS_NAMES)
}

print(f"{'Class':30s} | Weight | Triage")
print("-" * 60)
for idx, cls in enumerate(CLASS_NAMES):
    weight = class_weight_dict[idx]
    triage = TRIAGE_MAPPING[cls]
    print(f"{cls:30s} | {weight:6.1f} | {triage}")
print("=" * 60)

print("\nRationale:")
print("- Dataset is balanced at 250 images/class after augmentation")
print("- Base weight = 1.0 for all classes")
print("- REFER classes get 2.0x weight in Phase 1 (increased penalty for missed referrals)")
print("- REFER classes will get 3.0x weight in Phase 2 (further increased)")

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    """
    Focal Loss for addressing class imbalance and hard examples.
    
    Focal Loss = -alpha * (1 - p_t)^gamma * log(p_t)
    
    Args:
        gamma: Focusing parameter for hard examples (default: 2.0)
        alpha: Weighting factor for class imbalance (default: 0.25)
    
    Returns:
        Focal loss function
    """
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        cross_entropy = -y_true * tf.math.log(y_pred)
        loss = alpha * tf.pow(1.0 - y_pred, gamma) * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))
    return focal_loss_fixed

FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25

print("Focal Loss Configuration:")
print(f"  Gamma (focusing parameter): {FOCAL_GAMMA}")
print(f"  Alpha (balance parameter):  {FOCAL_ALPHA}")
print("\nFocal Loss addresses:")
print("  - Hard examples: gamma > 0 down-weights easy examples")
print("  - Class imbalance: alpha weights positive examples")
print("  - Combined with class_weight for comprehensive balance strategy")

In [ ]:
def create_model(num_classes=7, input_shape=(224, 224, 3), dropout_rate=0.4):
    """
    Create MobileNetV2-based classification model for skin lesion triage.
    
    Architecture:
        - MobileNetV2 base (ImageNet pretrained, frozen initially)
        - GlobalAveragePooling2D
        - Dense(256, relu) + L2 regularization + Dropout(0.4)
        - Dense(128, relu) + Dropout(0.2)
        - Dense(7, softmax)
    
    Args:
        num_classes: Number of output classes (default: 7)
        input_shape: Input image shape (default: 224x224x3)
        dropout_rate: Dropout rate for first dense layer (default: 0.4)
    
    Returns:
        model: Compiled Keras model
        base_model: MobileNetV2 base model (for layer unfreezing in Phase 2)
    """
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(dropout_rate / 2)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='DermoAI_MobileNetV2')
    return model, base_model

print("Creating model...")
model, base_model = create_model(
    num_classes=CONFIG['model']['num_classes'],
    input_shape=CONFIG['model']['input_shape'],
    dropout_rate=CONFIG['model']['dropout_rate']
)

print("\nModel Summary:")
model.summary()

trainable_params = np.sum([np.prod(v.get_shape()) for v in model.trainable_weights])
total_params = np.sum([np.prod(v.get_shape()) for v in model.weights])
print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters:     {total_params:,}")
print(f"Non-trainable:        {total_params - trainable_params:,}")

In [ ]:
print("Phase 1 Callbacks (Feature Extraction)")
print("=" * 60)

checkpoint_phase1 = ModelCheckpoint(
    filepath=str(MODEL_DIR / 'best_model.keras'),
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping_phase1 = EarlyStopping(
    monitor='val_loss',
    patience=CONFIG['callbacks']['early_stopping_patience'],
    restore_best_weights=True,
    verbose=1
)

reduce_lr_phase1 = ReduceLROnPlateau(
    monitor='val_loss',
    patience=CONFIG['callbacks']['reduce_lr_patience'],
    factor=CONFIG['callbacks']['reduce_lr_factor'],
    min_lr=CONFIG['callbacks']['min_lr'],
    verbose=1
)

csv_logger_phase1 = CSVLogger(
    str(RESULTS_DIR / 'training_history_phase1.csv'),
    append=False
)

callbacks_phase1 = [
    checkpoint_phase1,
    early_stopping_phase1,
    reduce_lr_phase1,
    csv_logger_phase1
]

print("Callbacks configured:")
for cb in callbacks_phase1:
    print(f"  - {cb.__class__.__name__}")
print("=" * 60)

In [ ]:
print("=" * 80)
print("PHASE 1: FEATURE EXTRACTION (Base Model Frozen)")
print("=" * 80)

model.compile(
    optimizer=Adam(learning_rate=CONFIG['training']['initial_lr']),
    loss=focal_loss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA),
    metrics=[
        'accuracy',
        Precision(name='precision'),
        Recall(name='recall'),
        AUC(name='auc')
    ]
)

print(f"\nOptimizer: Adam (lr={CONFIG['training']['initial_lr']})")
print(f"Loss: Focal Loss (gamma={FOCAL_GAMMA}, alpha={FOCAL_ALPHA})")
print(f"Metrics: accuracy, precision, recall, auc")
print(f"Epochs: {CONFIG['training']['phase1_epochs']}")
print(f"Batch size: {CONFIG['training']['batch_size']}")
print(f"Class weights: REFER classes = 2.0x")
print("\nStarting training...\n")

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG['training']['phase1_epochs'],
    callbacks=callbacks_phase1,
    class_weight=class_weight_dict,
    verbose=1
)

print("\n" + "=" * 80)
print("PHASE 1 COMPLETE")
print("=" * 80)
best_val_loss = min(history_phase1.history['val_loss'])
best_val_acc = max(history_phase1.history['val_accuracy'])
print(f"Best validation loss:     {best_val_loss:.4f}")
print(f"Best validation accuracy: {best_val_acc:.4f}")
print("=" * 80)

In [ ]:
print("=" * 80)
print("PHASE 2 SETUP: FINE-TUNING")
print("=" * 80)

print(f"\nBefore unfreezing:")
print(f"  Total layers in base_model: {len(base_model.layers)}")
print(f"  Trainable layers: {sum(1 for layer in base_model.layers if layer.trainable)}")

base_model.trainable = True
num_layers_to_unfreeze = 30
for layer in base_model.layers[:-num_layers_to_unfreeze]:
    layer.trainable = False

print(f"\nAfter unfreezing last {num_layers_to_unfreeze} layers:")
print(f"  Total layers in base_model: {len(base_model.layers)}")
print(f"  Trainable layers: {sum(1 for layer in base_model.layers if layer.trainable)}")

trainable_params = np.sum([np.prod(v.get_shape()) for v in model.trainable_weights])
total_params = np.sum([np.prod(v.get_shape()) for v in model.weights])
print(f"\nModel parameters:")
print(f"  Trainable: {trainable_params:,}")
print(f"  Total:     {total_params:,}")

class_weight_phase2 = {
    idx: 3.0 if cls in REFER_CLASSES else 1.0
    for idx, cls in enumerate(CLASS_NAMES)
}

print(f"\nClass weights for Phase 2:")
for idx, cls in enumerate(CLASS_NAMES):
    weight = class_weight_phase2[idx]
    triage = TRIAGE_MAPPING[cls]
    print(f"  {cls:30s} | {weight:4.1f}x | {triage}")

checkpoint_phase2 = ModelCheckpoint(
    filepath=str(MODEL_DIR / 'best_model.keras'),
    monitor='val_recall',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stopping_phase2 = EarlyStopping(
    monitor='val_recall',
    patience=CONFIG['callbacks']['early_stopping_patience'],
    restore_best_weights=True,
    mode='max',
    verbose=1
)

reduce_lr_phase2 = ReduceLROnPlateau(
    monitor='val_recall',
    patience=CONFIG['callbacks']['reduce_lr_patience'],
    factor=CONFIG['callbacks']['reduce_lr_factor'],
    min_lr=CONFIG['callbacks']['min_lr'],
    mode='max',
    verbose=1
)

csv_logger_phase2 = CSVLogger(
    str(RESULTS_DIR / 'training_history_phase2.csv'),
    append=False
)

callbacks_phase2 = [
    checkpoint_phase2,
    early_stopping_phase2,
    reduce_lr_phase2,
    csv_logger_phase2
]

model.compile(
    optimizer=Adam(learning_rate=CONFIG['training']['finetune_lr']),
    loss=focal_loss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA),
    metrics=[
        'accuracy',
        Precision(name='precision'),
        Recall(name='recall'),
        AUC(name='auc')
    ]
)

print(f"\nRecompiled with:")
print(f"  Learning rate: {CONFIG['training']['finetune_lr']} (reduced from {CONFIG['training']['initial_lr']})")
print(f"  Monitor metric: val_recall (changed from val_loss)")
print("=" * 80)

In [ ]:
print("=" * 80)
print("PHASE 2: FINE-TUNING (Last 30 Layers Unfrozen)")
print("=" * 80)

print(f"\nEpochs: {CONFIG['training']['phase2_epochs']}")
print(f"Learning rate: {CONFIG['training']['finetune_lr']}")
print(f"Class weights: REFER classes = 3.0x (increased from 2.0x)")
print("\nStarting fine-tuning...\n")

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG['training']['phase2_epochs'],
    callbacks=callbacks_phase2,
    class_weight=class_weight_phase2,
    verbose=1
)

print("\n" + "=" * 80)
print("PHASE 2 COMPLETE")
print("=" * 80)
best_val_recall = max(history_phase2.history['val_recall'])
best_val_acc = max(history_phase2.history['val_accuracy'])
print(f"Best validation recall:   {best_val_recall:.4f}")
print(f"Best validation accuracy: {best_val_acc:.4f}")
print("=" * 80)

In [ ]:
print("Plotting Training History")
print("=" * 60)

phase1_epochs = len(history_phase1.history['loss'])
phase2_epochs = len(history_phase2.history['loss'])
total_epochs = phase1_epochs + phase2_epochs

combined_history = {
    'loss': history_phase1.history['loss'] + history_phase2.history['loss'],
    'val_loss': history_phase1.history['val_loss'] + history_phase2.history['val_loss'],
    'accuracy': history_phase1.history['accuracy'] + history_phase2.history['accuracy'],
    'val_accuracy': history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy'],
    'recall': history_phase1.history['recall'] + history_phase2.history['recall'],
    'val_recall': history_phase1.history['val_recall'] + history_phase2.history['val_recall'],
    'auc': history_phase1.history['auc'] + history_phase2.history['auc'],
    'val_auc': history_phase1.history['val_auc'] + history_phase2.history['val_auc'],
}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('DermoAI Training History (Two-Phase)', fontsize=16, fontweight='bold')

metrics = [
    ('loss', 'Loss', axes[0, 0]),
    ('accuracy', 'Accuracy', axes[0, 1]),
    ('recall', 'Recall', axes[1, 0]),
    ('auc', 'AUC', axes[1, 1])
]

epochs_range = range(1, total_epochs + 1)

for metric_key, metric_name, ax in metrics:
    ax.plot(epochs_range, combined_history[metric_key], label=f'Train {metric_name}', linewidth=2)
    ax.plot(epochs_range, combined_history[f'val_{metric_key}'], label=f'Val {metric_name}', linewidth=2)
    ax.axvline(x=phase1_epochs, color='red', linestyle='--', linewidth=1.5, label='Phase Boundary')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(metric_name, fontsize=11)
    ax.set_title(f'{metric_name} over Training', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
history_plot_path = RESULTS_DIR / 'training_history.png'
plt.savefig(history_plot_path, dpi=150, bbox_inches='tight')
print(f"Training history plot saved to: {history_plot_path}")
plt.show()

In [ ]:
print("Loading Best Model and Generating Predictions")
print("=" * 60)

best_model_path = MODEL_DIR / 'best_model.keras'
print(f"Loading model from: {best_model_path}")

model = tf.keras.models.load_model(
    best_model_path,
    custom_objects={'focal_loss_fixed': focal_loss(FOCAL_GAMMA, FOCAL_ALPHA)}
)

print("Generating predictions on test set...")
test_predictions = model.predict(test_ds, verbose=1)

test_true_labels = np.concatenate([y.numpy() for x, y in test_ds], axis=0)
test_true_classes = np.argmax(test_true_labels, axis=1)
test_pred_classes = np.argmax(test_predictions, axis=1)

print(f"\nPrediction shape: {test_predictions.shape}")
print(f"Number of test samples: {len(test_true_classes)}")
print(f"Predictions generated successfully.")

In [ ]:
print("Overall Test Set Performance")
print("=" * 60)

test_accuracy = np.mean(test_pred_classes == test_true_classes)
test_recall_macro = recall_score(test_true_classes, test_pred_classes, average='macro')
test_precision_macro = precision_score(test_true_classes, test_pred_classes, average='macro')
test_f1_macro = f1_score(test_true_classes, test_pred_classes, average='macro')

print(f"\nAccuracy:        {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Macro Recall:    {test_recall_macro:.4f}")
print(f"Macro Precision: {test_precision_macro:.4f}")
print(f"Macro F1 Score:  {test_f1_macro:.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(
    test_true_classes,
    test_pred_classes,
    target_names=CLASS_NAMES,
    digits=4
))
print("=" * 60)

In [ ]:
print("Per-Class Performance Analysis")
print("=" * 80)

per_class_recall = recall_score(test_true_classes, test_pred_classes, average=None)
per_class_precision = precision_score(test_true_classes, test_pred_classes, average=None)
per_class_f1 = f1_score(test_true_classes, test_pred_classes, average=None)
per_class_support = np.bincount(test_true_classes, minlength=len(CLASS_NAMES))

performance_df = pd.DataFrame({
    'Class': CLASS_NAMES,
    'Recall': per_class_recall,
    'Precision': per_class_precision,
    'F1-Score': per_class_f1,
    'Support': per_class_support,
    'Triage': [TRIAGE_MAPPING[cls] for cls in CLASS_NAMES]
})

performance_df = performance_df.sort_values('Recall', ascending=True)

print(performance_df.to_string(index=False))
print("\n" + "=" * 80)

below_threshold = performance_df[performance_df['Recall'] < CONFIG['evaluation']['min_recall_threshold']]
if len(below_threshold) > 0:
    print(f"\n⚠️  WARNING: {len(below_threshold)} class(es) below {CONFIG['evaluation']['min_recall_threshold']:.0%} recall threshold:")
    for _, row in below_threshold.iterrows():
        print(f"  - {row['Class']}: {row['Recall']:.4f}")
else:
    print(f"\n✓ All classes meet minimum {CONFIG['evaluation']['min_recall_threshold']:.0%} recall threshold")

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(CLASS_NAMES))
width = 0.25

bars1 = ax.bar(x - width, performance_df['Recall'], width, label='Recall', alpha=0.8)
bars2 = ax.bar(x, performance_df['Precision'], width, label='Precision', alpha=0.8)
bars3 = ax.bar(x + width, performance_df['F1-Score'], width, label='F1-Score', alpha=0.8)

ax.axhline(y=0.50, color='red', linestyle='--', linewidth=1.5, label='Min Threshold (0.50)')
ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(performance_df['Class'], rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.0])

plt.tight_layout()
perf_plot_path = RESULTS_DIR / 'per_class_performance.png'
plt.savefig(perf_plot_path, dpi=150, bbox_inches='tight')
print(f"\nPer-class performance plot saved to: {perf_plot_path}")
plt.show()

In [ ]:
print("Applying Two-Stage Triage Decision Logic")
print("=" * 80)

CONFIDENCE_THRESHOLD = CONFIG['evaluation']['confidence_threshold']
REFER_THRESHOLD = 0.35

print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"REFER override threshold: {REFER_THRESHOLD}")
print("\nStage 1: Low confidence → route to best REFER class")
print("Stage 2: Any REFER class probability > threshold → force REFER prediction\n")

refer_indices = [CLASS_NAMES.index(c) for c in REFER_CLASSES]
max_confidences = np.max(test_predictions, axis=1)
test_pred_triage = np.argmax(test_predictions, axis=1).copy()

low_conf_mask = max_confidences < CONFIDENCE_THRESHOLD
refer_probs = test_predictions[:, refer_indices]
best_refer = [refer_indices[i] for i in np.argmax(refer_probs, axis=1)]
test_pred_triage[low_conf_mask] = np.array(best_refer)[low_conf_mask]

print(f"Stage 1: {np.sum(low_conf_mask)} predictions with confidence < {CONFIDENCE_THRESHOLD} routed to REFER")

stage2_overrides = 0
for r_idx in refer_indices:
    override_mask = test_predictions[:, r_idx] > REFER_THRESHOLD
    stage2_overrides += np.sum(override_mask & ~low_conf_mask)
    test_pred_triage[override_mask] = r_idx

print(f"Stage 2: {stage2_overrides} additional predictions overridden to REFER (probability > {REFER_THRESHOLD})")

refer_mask_true = np.isin(test_true_classes, refer_indices)
refer_mask_pred_original = np.isin(test_pred_classes, refer_indices)
refer_mask_pred_triage = np.isin(test_pred_triage, refer_indices)

refer_recall_original = recall_score(refer_mask_true, refer_mask_pred_original)
refer_recall_triage = recall_score(refer_mask_true, refer_mask_pred_triage)

print("\nREFER Recall Comparison:")
print(f"  Before thresholding: {refer_recall_original:.4f} ({refer_recall_original*100:.2f}%)")
print(f"  After thresholding:  {refer_recall_triage:.4f} ({refer_recall_triage*100:.2f}%)")

target_recall = CONFIG['evaluation']['refer_recall_target']
if refer_recall_triage >= target_recall:
    print(f"  ✓ PASS: Meets {target_recall:.0%} REFER recall target")
else:
    print(f"  ✗ FAIL: Below {target_recall:.0%} REFER recall target")

test_pred_classes = test_pred_triage
print(f"\nUsing thresholded predictions for all subsequent evaluations.")
print("=" * 80)

In [ ]:
print("Confusion Matrix (Thresholded Predictions)")
print("=" * 60)

cm = confusion_matrix(test_true_classes, test_pred_classes)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=ax,
    cbar_kws={'label': 'Count'}
)
ax.set_xlabel('Predicted Class', fontsize=12, fontweight='bold')
ax.set_ylabel('True Class', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix - Test Set (After Triage Thresholding)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

cm_plot_path = RESULTS_DIR / 'confusion_matrix.png'
plt.savefig(cm_plot_path, dpi=150, bbox_inches='tight')
print(f"Confusion matrix saved to: {cm_plot_path}")
plt.show()

In [ ]:
print("Critical Error Analysis: REFER → MANAGE LOCALLY Misclassifications")
print("=" * 80)

manage_locally_indices = [CLASS_NAMES.index(c) for c in CONFIG['evaluation']['manage_locally_classes']]

critical_errors_by_class = {}
total_critical_errors = 0

for refer_cls in REFER_CLASSES:
    refer_idx = CLASS_NAMES.index(refer_cls)
    
    refer_true_mask = test_true_classes == refer_idx
    pred_manage_mask = np.isin(test_pred_classes, manage_locally_indices)
    
    critical_errors = np.sum(refer_true_mask & pred_manage_mask)
    critical_errors_by_class[refer_cls] = critical_errors
    total_critical_errors += critical_errors
    
    print(f"{refer_cls:30s}: {critical_errors} critical error(s)")

print("-" * 80)
print(f"TOTAL CRITICAL ERRORS: {total_critical_errors}")

CRITICAL_ERROR_THRESHOLD = 3
if total_critical_errors <= CRITICAL_ERROR_THRESHOLD:
    print(f"✓ ACCEPTABLE: ≤{CRITICAL_ERROR_THRESHOLD} critical errors")
else:
    print(f"✗ TOO HIGH: >{CRITICAL_ERROR_THRESHOLD} critical errors")

print("\nNote: These are REFER cases incorrectly classified as MANAGE LOCALLY.")
print("Such errors pose the highest clinical risk (missed specialist referrals).")
print("=" * 80)

In [ ]:
print("REFER Class Recall Evaluation")
print("=" * 80)

refer_recall_by_class = {}

for refer_cls in REFER_CLASSES:
    refer_idx = CLASS_NAMES.index(refer_cls)
    refer_true_mask = test_true_classes == refer_idx
    refer_pred_mask = test_pred_classes == refer_idx
    
    if np.sum(refer_true_mask) > 0:
        cls_recall = np.sum(refer_true_mask & refer_pred_mask) / np.sum(refer_true_mask)
    else:
        cls_recall = 0.0
    
    refer_recall_by_class[refer_cls] = cls_recall
    print(f"{refer_cls:30s}: {cls_recall:.4f} ({cls_recall*100:.2f}%)")

refer_mask_true = np.isin(test_true_classes, refer_indices)
refer_mask_pred = np.isin(test_pred_classes, refer_indices)
combined_refer_recall = recall_score(refer_mask_true, refer_mask_pred)

print("-" * 80)
print(f"COMBINED REFER RECALL: {combined_refer_recall:.4f} ({combined_refer_recall*100:.2f}%)")

target_recall = CONFIG['evaluation']['refer_recall_target']
if combined_refer_recall >= target_recall:
    print(f"✓ PASS: Meets {target_recall:.0%} target")
else:
    print(f"✗ FAIL: Below {target_recall:.0%} target")

print("\nInterpretation:")
print(f"  - Combined REFER recall measures: of all true REFER cases, what % were")
print(f"    correctly identified as needing referral (any REFER class)")
print(f"  - Target: ≥{target_recall:.0%} for patient safety")
print("=" * 80)

In [ ]:
print("FST-Stratified Performance Evaluation")
print("=" * 80)

metadata_path = BASE_DIR / 'data' / 'raw' / 'fitzpatrick17k' / 'fitzpatrick17k_fst_v_vi.csv'

if metadata_path.exists():
    try:
        metadata_df = pd.read_csv(metadata_path)
        print(f"Loaded metadata from: {metadata_path}")
        
        test_filenames = []
        for images, labels in test_ds.unbatch():
            test_filenames.append("placeholder.jpg")
        
        test_image_paths = []
        for cls_idx in test_true_classes:
            cls_name = CLASS_NAMES[cls_idx]
            cls_dir = TEST_DIR / cls_name
            if cls_dir.exists():
                images = sorted(cls_dir.glob('*.jpg'))
                if len(test_image_paths) < len(test_true_classes):
                    test_image_paths.append(images[len([p for p in test_image_paths if cls_name in str(p)])] if images else None)
        
        test_fst_labels = []
        for img_path in test_image_paths:
            if img_path:
                img_filename = img_path.name
                match = metadata_df[metadata_df['md5hash'].apply(lambda x: img_filename.startswith(str(x)[:8]))]
                if not match.empty:
                    fst = match.iloc[0]['fitzpatrick']
                    test_fst_labels.append(fst)
                else:
                    test_fst_labels.append(None)
            else:
                test_fst_labels.append(None)
        
        fst_v_mask = np.array([fst == 5 for fst in test_fst_labels])
        fst_vi_mask = np.array([fst == 6 for fst in test_fst_labels])
        
        if np.sum(fst_v_mask) > 0 and np.sum(fst_vi_mask) > 0:
            fst_v_acc = np.mean(test_pred_classes[fst_v_mask] == test_true_classes[fst_v_mask])
            fst_vi_acc = np.mean(test_pred_classes[fst_vi_mask] == test_true_classes[fst_vi_mask])
            
            fst_v_recall = recall_score(test_true_classes[fst_v_mask], test_pred_classes[fst_v_mask], average='macro')
            fst_vi_recall = recall_score(test_true_classes[fst_vi_mask], test_pred_classes[fst_vi_mask], average='macro')
            
            fst_v_precision = precision_score(test_true_classes[fst_v_mask], test_pred_classes[fst_v_mask], average='macro')
            fst_vi_precision = precision_score(test_true_classes[fst_vi_mask], test_pred_classes[fst_vi_mask], average='macro')
            
            fst_v_f1 = f1_score(test_true_classes[fst_v_mask], test_pred_classes[fst_v_mask], average='macro')
            fst_vi_f1 = f1_score(test_true_classes[fst_vi_mask], test_pred_classes[fst_vi_mask], average='macro')
            
            print(f"\nFST V  (n={np.sum(fst_v_mask)}):")
            print(f"  Accuracy:  {fst_v_acc:.4f}")
            print(f"  Recall:    {fst_v_recall:.4f}")
            print(f"  Precision: {fst_v_precision:.4f}")
            print(f"  F1-Score:  {fst_v_f1:.4f}")
            
            print(f"\nFST VI (n={np.sum(fst_vi_mask)}):")
            print(f"  Accuracy:  {fst_vi_acc:.4f}")
            print(f"  Recall:    {fst_vi_recall:.4f}")
            print(f"  Precision: {fst_vi_precision:.4f}")
            print(f"  F1-Score:  {fst_vi_f1:.4f}")
            
            recall_gap = abs(fst_v_recall - fst_vi_recall)
            acc_gap = abs(fst_v_acc - fst_vi_acc)
            
            print(f"\nEquity Analysis:")
            print(f"  Recall gap:   {recall_gap:.4f} ({recall_gap*100:.2f}%)")
            print(f"  Accuracy gap: {acc_gap:.4f} ({acc_gap*100:.2f}%)")
            
            gap_threshold = CONFIG['evaluation']['fst_equity_gap_threshold']
            if recall_gap < gap_threshold:
                print(f"  ✓ PASS: Recall gap < {gap_threshold:.0%} threshold")
            else:
                print(f"  ✗ FAIL: Recall gap ≥ {gap_threshold:.0%} threshold")
            
            fig, ax = plt.subplots(figsize=(10, 6))
            metrics_names = ['Accuracy', 'Recall', 'Precision', 'F1-Score']
            fst_v_metrics = [fst_v_acc, fst_v_recall, fst_v_precision, fst_v_f1]
            fst_vi_metrics = [fst_vi_acc, fst_vi_recall, fst_vi_precision, fst_vi_f1]
            
            x = np.arange(len(metrics_names))
            width = 0.35
            
            bars1 = ax.bar(x - width/2, fst_v_metrics, width, label='FST V', alpha=0.8)
            bars2 = ax.bar(x + width/2, fst_vi_metrics, width, label='FST VI', alpha=0.8)
            
            ax.set_xlabel('Metric', fontsize=12, fontweight='bold')
            ax.set_ylabel('Score', fontsize=12, fontweight='bold')
            ax.set_title('Performance Comparison: FST V vs FST VI', fontsize=14, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels(metrics_names)
            ax.legend(fontsize=11)
            ax.grid(True, alpha=0.3, axis='y')
            ax.set_ylim([0, 1.0])
            
            plt.tight_layout()
            fst_plot_path = RESULTS_DIR / 'fst_comparison.png'
            plt.savefig(fst_plot_path, dpi=150, bbox_inches='tight')
            print(f"\nFST comparison plot saved to: {fst_plot_path}")
            plt.show()
        else:
            print("Insufficient FST V or FST VI samples for stratified analysis.")
            recall_gap = None
    
    except Exception as e:
        print(f"Error loading or processing metadata: {e}")
        print("Skipping FST-stratified evaluation.")
        recall_gap = None
else:
    print(f"Metadata file not found: {metadata_path}")
    print("Skipping FST-stratified evaluation.")
    recall_gap = None

print("=" * 80)

In [ ]:
print("Prediction Summary by Class")
print("=" * 80)

print(f"{'Class':30s} | Correct | Total | Accuracy")
print("-" * 80)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_mask = test_true_classes == cls_idx
    cls_correct = np.sum((test_pred_classes == cls_idx) & cls_mask)
    cls_total = np.sum(cls_mask)
    cls_acc = cls_correct / cls_total if cls_total > 0 else 0.0
    
    print(f"{cls_name:30s} | {cls_correct:7d} | {cls_total:5d} | {cls_acc:.4f}")

print("=" * 80)

print("\nConfidence Distribution:")
confidence_stats = {
    'Mean confidence': np.mean(max_confidences),
    'Median confidence': np.median(max_confidences),
    'Min confidence': np.min(max_confidences),
    'Max confidence': np.max(max_confidences),
    'Below 0.50 (→ REFER)': np.sum(max_confidences < 0.50) / len(max_confidences),
    'Above 0.80 (high confidence)': np.sum(max_confidences > 0.80) / len(max_confidences)
}

for stat_name, stat_value in confidence_stats.items():
    if 'Below' in stat_name or 'Above' in stat_name:
        print(f"  {stat_name}: {stat_value:.2%}")
    else:
        print(f"  {stat_name}: {stat_value:.4f}")

print("\nInterpretation:")
print("  - Low confidence predictions (< 0.50) are routed to REFER as UNCERTAIN")
print("  - High confidence predictions (> 0.80) indicate strong model certainty")
print("=" * 80)

In [ ]:
print("Success Criteria Evaluation")
print("=" * 80)

all_classes_above_min = all(per_class_recall >= CONFIG['evaluation']['min_recall_threshold'])

criteria = {
    '1. Overall accuracy ≥70%': test_accuracy >= 0.70,
    '2. Overall accuracy ≥80% (stretch target)': test_accuracy >= 0.80,
    '3. All classes recall ≥50%': all_classes_above_min,
    '4. REFER recall ≥75%': combined_refer_recall >= 0.75,
    '5. Critical errors (REFER→MANAGE LOCALLY) ≤3': total_critical_errors <= 3,
    '6. FST equity gap <10%': recall_gap < 0.10 if recall_gap is not None else None
}

passed_criteria = 0
total_criteria = len([v for v in criteria.values() if v is not None])

for criterion, result in criteria.items():
    if result is None:
        status = "N/A"
    elif result:
        status = "✓ PASS"
        passed_criteria += 1
    else:
        status = "✗ FAIL"
    
    print(f"{criterion:50s} : {status}")

print("=" * 80)
print(f"\nOVERALL SUMMARY: {passed_criteria}/{total_criteria} criteria passed")

if passed_criteria == total_criteria:
    print("🎉 All success criteria met! Model ready for deployment.")
elif passed_criteria >= total_criteria * 0.75:
    print("⚠️  Most criteria met. Review failing criteria before deployment.")
else:
    print("❌ Multiple criteria not met. Further training/tuning recommended.")

print("=" * 80)

In [ ]:
print("Saving Model Artifacts")
print("=" * 80)

final_model_path = MODEL_DIR / 'dermoai_final_model.keras'
model.save(final_model_path)
print(f"✓ Model saved: {final_model_path}")

class_names_path = MODEL_DIR / 'class_names.json'
with open(class_names_path, 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)
print(f"✓ Class names saved: {class_names_path}")

triage_mapping_path = MODEL_DIR / 'triage_mapping.json'
with open(triage_mapping_path, 'w') as f:
    json.dump(TRIAGE_MAPPING, f, indent=2)
print(f"✓ Triage mapping saved: {triage_mapping_path}")

config_path = MODEL_DIR / 'training_config.json'
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2)
print(f"✓ Training config saved: {config_path}")

history_csv_path = RESULTS_DIR / 'training_history_combined.csv'
combined_history_df = pd.DataFrame({
    'epoch': list(range(1, total_epochs + 1)),
    'loss': combined_history['loss'],
    'val_loss': combined_history['val_loss'],
    'accuracy': combined_history['accuracy'],
    'val_accuracy': combined_history['val_accuracy'],
    'recall': combined_history['recall'],
    'val_recall': combined_history['val_recall'],
    'auc': combined_history['auc'],
    'val_auc': combined_history['val_auc'],
    'phase': ['Phase 1'] * phase1_epochs + ['Phase 2'] * phase2_epochs
})
combined_history_df.to_csv(history_csv_path, index=False)
print(f"✓ Training history saved: {history_csv_path}")

model_card_path = MODEL_DIR / 'MODEL_CARD.md'
model_card_content = f"""# DermoAI Model Card

## Model Details

**Model Name:** DermoAI MobileNetV2 FST V-VI Skin Lesion Classifier v2  
**Architecture:** MobileNetV2 (ImageNet pretrained) + Custom Classification Head  
**Framework:** TensorFlow/Keras  
**Date:** {datetime.now().strftime('%Y-%m-%d')}  
**Version:** 2.0

## Intended Use

**Primary Use:** AI-assisted dermatological triage for resource-limited primary care settings in Rwanda

**Target Population:** Patients with Fitzpatrick Skin Types (FST) V-VI (darker skin tones)

**Clinical Workflow:** 
1. Primary care provider captures skin lesion image
2. Model classifies into one of 7 conditions
3. System routes to either REFER (specialist) or MANAGE LOCALLY (primary care)

## Model Architecture

- **Base Model:** MobileNetV2 (ImageNet weights)
- **Input:** 224×224×3 RGB images
- **Classification Head:**
  - GlobalAveragePooling2D
  - Dense(256, relu) + L2 regularization + Dropout(0.4)
  - Dense(128, relu) + Dropout(0.2)
  - Dense(7, softmax)
- **Parameters:** {total_params:,} total ({trainable_params:,} trainable after fine-tuning)

## Training Strategy

**Two-Phase Training:**

1. **Phase 1 — Feature Extraction (20 epochs)**
   - Base model frozen
   - Learning rate: 1e-3
   - REFER class weight: 2.0x

2. **Phase 2 — Fine-Tuning (15 epochs)**
   - Last 30 layers unfrozen
   - Learning rate: 1e-5
   - REFER class weight: 3.0x

**Loss Function:** Focal Loss (gamma=2.0, alpha=0.25)

**Optimization:** Adam optimizer with ReduceLROnPlateau

## 7 Condition Classes and Triage Mapping

| Condition | Triage Decision |
|-----------|----------------|
| lichen_planus | MANAGE LOCALLY |
| lupus_erythematosus | REFER |
| pityriasis_rubra_pilaris | REFER |
| psoriasis | MANAGE LOCALLY |
| scabies | MANAGE LOCALLY |
| squamous_cell_carcinoma | REFER |
| vitiligo | MANAGE LOCALLY |

## Two-Stage Triage Logic

**Stage 1:** If max confidence < 0.50 → route to best REFER class (UNCERTAIN)

**Stage 2:** If any REFER class probability > 0.35 → force REFER prediction

**Rationale:** Conservative approach prioritizes patient safety by defaulting uncertain cases to specialist referral

## Training Dataset

**Source:** Fitzpatrick17k dataset, FST V-VI subset

**Distribution:** Perfectly balanced at 250 images per class (1,750 total training images)

**Augmentation:** Applied to balance classes and improve generalization

**Validation/Test:** Natural distribution from Fitzpatrick17k FST V-VI images

## Performance Metrics

### Overall Performance
- **Accuracy:** {test_accuracy:.4f} ({test_accuracy*100:.2f}%)
- **Macro Recall:** {test_recall_macro:.4f}
- **Macro Precision:** {test_precision_macro:.4f}
- **Macro F1-Score:** {test_f1_macro:.4f}

### REFER Class Performance
- **Combined REFER Recall:** {combined_refer_recall:.4f} ({combined_refer_recall*100:.2f}%)
- **Target:** ≥75% (patient safety critical)

### Critical Errors
- **REFER→MANAGE LOCALLY misclassifications:** {total_critical_errors}
- **Target:** ≤3

### FST Equity
- **Recall Gap (FST V vs VI):** {recall_gap:.4f if recall_gap is not None else 'N/A'}
- **Target:** <10%

## Limitations

1. **Training Data:** Model trained exclusively on FST V-VI images; may not generalize to lighter skin types
2. **Image Quality:** Performance assumes good lighting, focus, and framing
3. **Clinical Context:** Model does not consider patient history, symptoms, or other clinical factors
4. **7 Conditions Only:** Cannot detect conditions outside the 7 trained classes
5. **Geographic Specificity:** Optimized for Rwanda healthcare context; may need adaptation elsewhere
6. **Not Diagnostic:** Intended for triage support only, not definitive diagnosis

## Ethical Considerations

- **Bias Mitigation:** Specifically addresses AI performance gap on darker skin tones
- **Clinical Validation Required:** Model should be validated in real clinical settings before deployment
- **Human Oversight:** All predictions should be reviewed by qualified healthcare providers
- **Transparency:** Confidence scores provided to support clinical decision-making

## Maintenance and Monitoring

- **Retraining:** Recommended annually or when performance degrades
- **Data Drift:** Monitor for changes in image quality, demographics, or condition prevalence
- **Performance Monitoring:** Track REFER recall and critical errors in production

## Contact

For questions or issues, contact the DermoAI development team.

---

*This model card follows guidelines from Mitchell et al. (2019) and is intended to promote transparency and responsible AI deployment.*
"""

with open(model_card_path, 'w') as f:
    f.write(model_card_content)
print(f"✓ Model card saved: {model_card_path}")

print("=" * 80)
print("All artifacts saved successfully!")
print("=" * 80)

In [ ]:
print("Saving Final Report JSON")
print("=" * 80)

final_report = {
    'model': {
        'name': 'DermoAI_MobileNetV2_v2',
        'architecture': CONFIG['model']['architecture'],
        'input_shape': CONFIG['model']['input_shape'],
        'num_classes': CONFIG['model']['num_classes'],
        'total_parameters': int(total_params),
        'trainable_parameters': int(trainable_params),
    },
    'training': {
        'strategy': 'two_phase',
        'phase1_epochs': phase1_epochs,
        'phase2_epochs': phase2_epochs,
        'total_epochs': total_epochs,
        'focal_loss': {'gamma': FOCAL_GAMMA, 'alpha': FOCAL_ALPHA},
        'batch_size': CONFIG['training']['batch_size'],
        'initial_lr': CONFIG['training']['initial_lr'],
        'finetune_lr': CONFIG['training']['finetune_lr'],
        'class_weights_phase1': {CLASS_NAMES[idx]: weight for idx, weight in class_weight_dict.items()},
        'class_weights_phase2': {CLASS_NAMES[idx]: weight for idx, weight in class_weight_phase2.items()},
    },
    'dataset': {
        'train_samples': int(df_dist['Train'].sum()),
        'val_samples': int(df_dist['Val'].sum()),
        'test_samples': int(df_dist['Test'].sum()),
        'train_distribution': df_dist[['Class', 'Train']].set_index('Class').to_dict()['Train'],
        'balanced': float(imbalance_ratio) <= 1.1,
    },
    'performance': {
        'overall': {
            'accuracy': float(test_accuracy),
            'macro_recall': float(test_recall_macro),
            'macro_precision': float(test_precision_macro),
            'macro_f1': float(test_f1_macro),
        },
        'per_class': {
            CLASS_NAMES[i]: {
                'recall': float(per_class_recall[i]),
                'precision': float(per_class_precision[i]),
                'f1': float(per_class_f1[i]),
                'support': int(per_class_support[i]),
            }
            for i in range(len(CLASS_NAMES))
        },
        'refer': {
            'combined_recall': float(combined_refer_recall),
            'target': CONFIG['evaluation']['refer_recall_target'],
            'meets_target': bool(combined_refer_recall >= CONFIG['evaluation']['refer_recall_target']),
            'by_class': {cls: float(recall) for cls, recall in refer_recall_by_class.items()},
        },
        'critical_errors': {
            'total': int(total_critical_errors),
            'by_class': {cls: int(count) for cls, count in critical_errors_by_class.items()},
            'threshold': 3,
            'acceptable': bool(total_critical_errors <= 3),
        },
        'fst_equity': {
            'recall_gap': float(recall_gap) if recall_gap is not None else None,
            'threshold': CONFIG['evaluation']['fst_equity_gap_threshold'],
            'meets_target': bool(recall_gap < 0.10) if recall_gap is not None else None,
        }
    },
    'triage': {
        'confidence_threshold': CONFIG['evaluation']['confidence_threshold'],
        'refer_override_threshold': 0.35,
        'low_confidence_routed': int(np.sum(low_conf_mask)),
        'refer_override_count': int(stage2_overrides),
        'mapping': TRIAGE_MAPPING,
    },
    'success_criteria': {
        criterion: bool(result) if result is not None else None
        for criterion, result in criteria.items()
    },
    'artifacts': {
        'model_path': str(final_model_path),
        'class_names_path': str(class_names_path),
        'triage_mapping_path': str(triage_mapping_path),
        'config_path': str(config_path),
        'model_card_path': str(model_card_path),
    },
    'metadata': {
        'timestamp': datetime.now().isoformat(),
        'tensorflow_version': tf.__version__,
        'random_seed': RANDOM_SEED,
    }
}

report_path = RESULTS_DIR / 'final_report.json'
with open(report_path, 'w') as f:
    json.dump(final_report, f, indent=2)

print(f"✓ Final report saved: {report_path}")
print("\nAll saved files:")
print(f"  1. {final_model_path}")
print(f"  2. {class_names_path}")
print(f"  3. {triage_mapping_path}")
print(f"  4. {config_path}")
print(f"  5. {model_card_path}")
print(f"  6. {history_csv_path}")
print(f"  7. {report_path}")
print(f"  8. {history_plot_path}")
print(f"  9. {perf_plot_path}")
print(f" 10. {cm_plot_path}")
if recall_gap is not None:
    print(f" 11. {fst_plot_path}")

print("=" * 80)

# Training Complete — Summary

## What Was Built

This notebook implemented a **MobileNetV2-based skin lesion classifier** optimized for Fitzpatrick Skin Types V-VI (darker skin tones) with a focus on clinical triage for Rwanda primary care settings.

### Architecture
- **Base Model:** MobileNetV2 pretrained on ImageNet
- **Classification Head:** Two dense layers (256 → 128) with dropout regularization
- **Output:** 7-class softmax classifier

### 7 Condition Classes

| Condition | Triage Decision |
|-----------|----------------|
| lichen_planus | MANAGE LOCALLY |
| lupus_erythematosus | **REFER** |
| pityriasis_rubra_pilaris | **REFER** |
| psoriasis | MANAGE LOCALLY |
| scabies | MANAGE LOCALLY |
| squamous_cell_carcinoma | **REFER** |
| vitiligo | MANAGE LOCALLY |

### Two-Phase Training Approach

**Phase 1 (Feature Extraction):**
- Base model frozen to preserve ImageNet features
- Higher learning rate (1e-3) for fast classifier training
- REFER classes weighted 2.0x

**Phase 2 (Fine-Tuning):**
- Last 30 layers unfrozen for task-specific adaptation
- Lower learning rate (1e-5) for careful weight updates
- REFER classes weighted 3.0x (increased clinical penalty)

### Focal Loss Rationale

**Focal Loss (gamma=2.0, alpha=0.25)** addresses two challenges:
1. **Hard Examples:** Gamma parameter down-weights easy examples, forcing model to focus on difficult cases
2. **Class Balance:** Alpha parameter adjusts weighting for positive examples

Combined with class weights, this creates a comprehensive strategy to prevent REFER class under-performance.

### Two-Stage Triage Logic

**Stage 1 — Low Confidence Handling:**
- Predictions with confidence < 0.50 are considered **UNCERTAIN**
- Routed to best REFER class (conservative safety default)

**Stage 2 — REFER Safety Net:**
- If ANY REFER class probability exceeds 0.35, force REFER prediction
- Catches cases where model is moderately confident about a serious condition

**Rationale:** Missing a serious condition (false negative REFER) has much higher clinical cost than an unnecessary referral (false positive REFER).

## Performance Achieved

- **Overall Accuracy:** See Cell 18 output
- **REFER Recall:** See Cell 23 output (target: ≥75%)
- **Critical Errors:** See Cell 22 output (target: ≤3)
- **FST Equity Gap:** See Cell 24 output (target: <10%)

## Next Steps

**Notebook 05 — Model Evaluation:**
- Comprehensive error analysis on misclassified cases
- Confidence calibration assessment
- Clinical risk stratification
- Deployment readiness checklist

**Deployment Pipeline:**
- Convert to TensorFlow Lite for mobile deployment
- Build FastAPI inference server
- Create Next.js PWA frontend
- Set up production monitoring

---

**Model v2 Training Complete** ✓